# Mari original synthetic voice casting — compatibility fix v03

Install Chatterbox's pinned Torch/Torchaudio into a fresh Kaggle process. Remove Kaggle's incompatible optional torchvision version before verifying the Transformers `LlamaModel` import. No model weights are downloaded until preflight passes. All WAVs remain unapproved.


In [ ]:
import os,sys,subprocess,json
from pathlib import Path
os.environ["HF_HOME"]="/kaggle/temp/ninefold_mari_chatterbox_models"
os.environ["TORCH_HOME"]="/kaggle/temp/ninefold_mari_chatterbox_torch"
# Do NOT import torch in this installer Python process. Pip may replace torch
# from Kaggle's preinstalled version; loading stale torch C libraries caused
# the previous libtorchaudio.so error in the same running interpreter.
p=subprocess.run([sys.executable,"-m","pip","install","-q",
                  "--disable-pip-version-check","chatterbox-tts","soundfile"],
                 capture_output=True,text=True,timeout=850)
print("Chatterbox dependencies installed:",p.returncode==0,flush=True)
if p.returncode:
    print("Install error excerpt:",(p.stderr or "")[-1000:])
    raise RuntimeError("Chatterbox installation failed; no TTS generation started")
# Kaggle provides torchvision 0.26+cu128, built for torch 2.11, but
# Chatterbox pins torch 2.6. Transformers' lazy LlamaModel import breaks if
# the incompatible optional image torchvision package remains installed.
# We only generate AUDIO here: remove this unnecessary optional package in
# this disposable private Kaggle session (not user's other notebooks).
remove=subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision"],
    capture_output=True,text=True,timeout=120)
print("Removed optional incompatible torchvision:",remove.returncode==0,flush=True)
# Fail *before* downloading GBs of model weights if the library import breaks.
preflight=subprocess.run([sys.executable,"-c",
    "import torch,torchaudio; from transformers import LlamaModel; print('Torch:',torch.__version__,'Torchaudio:',torchaudio.__version__,'LlamaModel import OK')"],
    capture_output=True,text=True,timeout=130)
Path("/kaggle/working/mari_chatterbox_v03_preflight.log").write_text(
    (preflight.stdout or "")[-2000:]+"\\n-- STDERR --\\n"+(preflight.stderr or "")[-6500:],encoding="utf8")
print("Torch/Torchaudio/Transformers preflight:",preflight.returncode==0,flush=True)
if preflight.returncode:
    print("Preflight traceback:",(preflight.stderr or "")[-1500:],flush=True)
    raise RuntimeError("Voice model dependency preflight did not pass; no model weights downloaded")
# Execute generation in a NEW interpreter that imports only matched torch and
# torchaudio binaries from the just-installed Chatterbox 0.1.7 dependencies.
worker_path=Path("/kaggle/temp/ninefold_mari_chatterbox_v03_worker.py")
worker_path.parent.mkdir(parents=True,exist_ok=True)
worker_path.write_text("import os,sys,json,subprocess,hashlib,time,traceback,zipfile\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nos.environ[\"HF_HOME\"]=\"/kaggle/temp/ninefold_mari_chatterbox_models\"\nos.environ[\"TORCH_HOME\"]=\"/kaggle/temp/ninefold_mari_chatterbox_torch\"\nroot=Path(\"/kaggle/working/ninefold_mari_original_chatterbox\")\nroot.mkdir(exist_ok=True,parents=True)\nstatus={\"character\":\"Mari\",\"age\":6,\"model\":\"ResembleAI/Chatterbox original\",\n        \"approval\":\"UNAPPROVED_ORIGINAL_SYNTHETIC_VOICE_CANDIDATES\",\n        \"state\":\"STARTED\",\"completed_steps\":[],\"candidate_count\":0,\n        \"started_utc\":datetime.now(timezone.utc).isoformat(),\"files\":[]}\ndef save(): (root/\"mari_recast_status.json\").write_text(json.dumps(status,indent=2)+\"\\n\")\ndef sha(b): return hashlib.sha256(b).hexdigest()\nsave()\ntry:\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Free Kaggle CUDA GPU required; no CPU fallback\")\n    free,total=torch.cuda.mem_get_info()\n    if free<5*1024**3:\n        raise RuntimeError(\"Not enough free GPU memory\")\n    status[\"completed_steps\"].append(\"GPU preflight\")\n    source_files=list(Path(\"/kaggle/input\").rglob(\"mari_voice_01_UNAPPROVED.wav\"))\n    source_zips=list(Path(\"/kaggle/input\").rglob(\"ninefold_17_supporting_cast_UNAPPROVED.zip\"))\n    if len(source_files)==1:\n        base_audio=source_files[0].read_bytes()\n    elif len(source_zips)==1:\n        with zipfile.ZipFile(source_zips[0]) as z:\n            base_audio=z.read(\"mari_voice_01_UNAPPROVED.wav\")\n    else:\n        raise RuntimeError(\"Missing or duplicate trusted Mari original synthetic reference; check Kaggle kernel_sources\")\n    ref=root/\"mari_original_synthetic_reference_UNAPPROVED.wav\"\n    ref.write_bytes(base_audio)\n    import wave\n    with wave.open(str(ref),\"rb\") as wav:\n        if wav.getframerate()<22050 or wav.getsampwidth()!=2 or wav.getnframes()==0:\n            raise ValueError(\"Unexpected reference PCM format\")\n    status[\"source_ref_sha256\"]=sha(base_audio)\n    status[\"completed_steps\"].append(\"Original Mari synthetic reference loaded from private Kaggle notebook source\")\n    # Dependencies installed before this separate Python worker is started.\n    status[\"completed_steps\"].append(\"Torch/torchaudio loaded in new process after compatible installation\")\n    from chatterbox.tts import ChatterboxTTS\n    import soundfile as sf\n    model=ChatterboxTTS.from_pretrained(device=\"cuda\")\n    status[\"completed_steps\"].append(\"Loaded Chatterbox original\")\n    scripts=[\n      (\"I can see the step. Please, hold my hand.\",0.35,0.30),\n      (\"Wait. Please don't go. I'm scared.\",0.50,0.35),\n      (\"Can you help me? I can't jump.\",0.60,0.30),\n      (\"You promised you'd hold my hand.\",0.42,0.45)\n    ]\n    for i,(text,exaggeration,cfg) in enumerate(scripts,1):\n        seed=20261031+211*i\n        torch.manual_seed(seed)\n        wav=model.generate(text,audio_prompt_path=str(ref),\n                  exaggeration=exaggeration,cfg_weight=cfg)\n        signal=wav.squeeze().detach().float().cpu().numpy()\n        dest=root/f\"mari_chatterbox_{i:02d}_UNAPPROVED.wav\"\n        sf.write(str(dest),signal,model.sr,subtype=\"PCM_16\")\n        with wave.open(str(dest),\"rb\") as f:\n            dur=f.getnframes()/f.getframerate()\n        if dur<0.5 or dur>18:raise RuntimeError(\"Invalid generation duration\")\n        status[\"files\"].append({\"file\":dest.name,\"text\":text,\n            \"duration_seconds\":round(dur,3),\"seed\":seed,\"exaggeration\":exaggeration,\n            \"cfg_weight\":cfg,\"sha256\":sha(dest.read_bytes()),\n            \"human_reviewed\":False,\"age_authenticity_approved\":False})\n        status[\"candidate_count\"]=len(status[\"files\"])\n        save()\n        print(\"MARI_CHATTERBOX_CANDIDATE\",i,round(dur,2),flush=True)\n    status[\"state\"]=\"ALL_FOUR_ORIGINAL_RETAKES_GENERATED\"\nexcept Exception as exc:\n    status[\"state\"]=\"FAILED_OR_PARTIAL\"\n    status[\"error\"]=type(exc).__name__+\": \"+str(exc)[:300]\n    print(\"MARI_CHATTERBOX_ERROR\",status[\"error\"],flush=True)\nfinally:\n    status[\"ended_utc\"]=datetime.now(timezone.utc).isoformat()\n    save()\n    archive=Path(\"/kaggle/working/ninefold_mari_chatterbox_UNAPPROVED.zip\")\n    with zipfile.ZipFile(archive,\"w\",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:\n        for file in root.iterdir():\n            if file.suffix.lower() in (\".wav\",\".json\",\".txt\"):\n                z.write(file,arcname=file.name)\n    print(\"MARI_CHATTERBOX_FINAL\",status[\"state\"],status[\"candidate_count\"],\n          archive.stat().st_size,flush=True)\n",encoding="utf8")
print("Executing voice worker in fresh interpreter; GPU quota unchanged by kernel split",flush=True)
run=subprocess.run([sys.executable,str(worker_path)],capture_output=True,text=True,timeout=1600,env=os.environ.copy())
out=Path("/kaggle/working/ninefold_mari_original_chatterbox")
out.mkdir(parents=True,exist_ok=True)
(out/"mari_chatterbox_v03_worker.log").write_text((run.stdout or "")[-20000:]+"\n--- STDERR ---\n"+(run.stderr or "")[-12000:],encoding="utf8")
print((run.stdout or "")[-3500:],flush=True)
if run.returncode:
    print("Worker failed:",(run.stderr or "")[-1500:],flush=True)
    raise RuntimeError("Fresh Chatterbox worker failed (inspect zipped output/worker log)")
